# Reproduce every number and figure
Run top to bottom in Google Colab. You need a Google Cloud project registered for non-commercial Earth Engine use. Nothing is exported; only pixels at sample points are read.

In [ ]:
PROJECT = "your-cloud-project-id"
!git clone -q https://github.com/abboudberoual/alphaearth-airport-solar /content/proj
import sys, os, shutil; sys.path.insert(0, "/content/proj/src")
import ee; ee.Authenticate(); ee.Initialize(project=PROJECT)

## Stage 2: labels and features
To reuse the exact label snapshot behind the published results, copy `data/uspvdb_snapshot_*.geojson.gz` from the repo into `/content/data` first; otherwise a fresh snapshot is downloaded and numbers can differ slightly.

In [ ]:
os.makedirs("/content/data", exist_ok=True)
for f in os.listdir("/content/proj/data"):
    if f.startswith("uspvdb_snapshot_"): shutil.copy(f"/content/proj/data/{f}", "/content/data/")
from aes import stage2
stage2.run("/content/data", s2_workers=16)

## Stages 3-4: label-efficiency experiment

In [ ]:
from aes import train
shutil.copy("/content/data/stage2_report.json", "/content/results/stage2_report.json") if os.path.isdir("/content/results") else (os.makedirs("/content/results"), shutil.copy("/content/data/stage2_report.json", "/content/results/stage2_report.json"))
train.run("/content/data", "/content/results")

## Stage 5: airports

In [ ]:
from aes import airports
airports.run("/content/proj/data/airports_top20.csv", "/content/results/final_model.json", "/content/data", "/content/results/airports.csv")

## Rebuild the page data
Copies results into the repo layout and regenerates `docs/data/*.json`.

In [ ]:
os.chdir("/content/proj"); shutil.copytree("/content/results", "results", dirs_exist_ok=True)
from aes import site_data; site_data.build()